# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed and up-to-date
!pip install --upgrade mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id` values using the Croissant schema.

Each entity (record set, field, column, etc.) is referenced by its `@id` for consistency.

In [ ]:
# Preview the record sets available via their @id
record_sets = list(dataset.record_sets)
print('Available record sets and their @id:')
for record_set in record_sets:
    print(f"- {record_set['@id']} ({record_set.get('name', '')})")

# For each record set, show its fields (columns) and their @id
for record_set in record_sets:
    print(f"\nFields in record set '{record_set.get('name', '')}' ({record_set['@id']}):")
    fields = record_set.get('field', [])
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field['@id']}: {field.get('name', field['@id'])}")
        else:
            print(f"  - {field}")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. Reference each record set and field by their `@id`.

You can analyze the DataFrame structure for each record set.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load records for each record set as a DataFrame
for rs_id in record_set_ids:
    # Use generator, then create DataFrame
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nColumns in record set '{rs_id}': {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"\nRecord set '{rs_id}' has no records loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter records based on a field, normalize numeric values, group by categorical attribute, and remove outliers.

All field references are made by their `@id`.

We'll demonstrate EDA for the main tabular record set. If unsure, choose the first available set.

In [ ]:
# Select the primary tabular record set for demonstration
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id}\nColumns: {df.columns.tolist()}")

    # Identify a likely numeric field (e.g., 'Age', referenced by its @id)
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id is None:
        numeric_field_id = df.columns[0]  # Fall back: use first column

    print(f"Numeric field selected: {numeric_field_id}")

    # Filter for values above threshold (e.g., Age > 50)
    threshold = 50
    if np.issubdtype(df[numeric_field_id].dtype, np.number):
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a key field (e.g., 'Sex' or 'MSI Status') by its @id
        group_field_id = None
        for col in df.columns:
            if 'sex' in col.lower() or 'msi' in col.lower():
                group_field_id = col
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} is not numeric, skipping EDA.")
else:
    print("No DataFrame found to perform EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Reference fields by their `@id`.

In [ ]:
# Visualize numeric field distributions and group comparisons
if 'filtered_df' in locals() and len(filtered_df) > 0:
    plt.figure(figsize=(7, 4))
    plt.hist(filtered_df[numeric_field_id], bins=10, color='skyblue', edgecolor='k', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} in Filtered Records")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No filtered data available for visualization.')

## 6. Conclusion

This notebook demonstrated loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

- Data was referenced throughout by `@id`, ensuring reproducibility and clarity.
- The loaded tabular data supports clinical investigations of colorectal cancer biomarkers in survivors.
- The EDA section illustrated filtering records, normalization, and grouping using relevant fields.
- Visualizations reveal numeric and categorical patterns for further model development.

You can extend this notebook to tailor analyses for biomedical or health informatics tasks using the FAIR^2 schema.